# Vanilla PCA + DBSCAN for PSD Signature Detection

## Problem Statement

We have Power Spectral Density (PSD) plots collected over time:
- Each PSD: 14,000 frequency points × 14,000 power values
- ~8000 PSDs collected over a week (one per timestamp)
- **Goal**: Identify recurring patterns ("signatures") without prior knowledge
- **Goal**: Distinguish normal behavior from anomalies

## Approach

1. Generate synthetic PSD data with known patterns
2. Apply PCA for dimensionality reduction
3. Use DBSCAN for clustering
4. Visualize clusters and representative traces

## Examples
1. Clean synthetic data with distinct signatures
2. Perturbed data with noise to test robustness

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
import seaborn as sns
from typing import Tuple, List, Dict
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Set plot style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

## Function Definitions

In [ ]:
def generate_synthetic_psd_signatures(n_frequencies: int = 14000) -> Tuple[np.ndarray, List[callable]]:
    """
    Generate base signature patterns for synthetic PSD data.
    
    Args:
        n_frequencies: Number of frequency points
        
    Returns:
        frequencies: Array of frequency values
        signature_functions: List of functions that generate different PSD patterns
    """
    frequencies = np.linspace(0, 6000, n_frequencies)  # 0 to 6 GHz
    
    # Define different signature patterns
    def signature_1(freqs):
        """Flat baseline with narrow peak at 1 GHz"""
        baseline = -80 * np.ones_like(freqs)
        peak = 30 * np.exp(-((freqs - 1000)**2) / (50**2))
        return baseline + peak
    
    def signature_2(freqs):
        """Flat baseline with two peaks at 2 GHz and 4 GHz"""
        baseline = -80 * np.ones_like(freqs)
        peak1 = 25 * np.exp(-((freqs - 2000)**2) / (80**2))
        peak2 = 25 * np.exp(-((freqs - 4000)**2) / (80**2))
        return baseline + peak1 + peak2
    
    def signature_3(freqs):
        """Sloped baseline with broad peak at 3 GHz"""
        baseline = -85 + (freqs / 6000) * 10  # Slight upward slope
        peak = 35 * np.exp(-((freqs - 3000)**2) / (200**2))
        return baseline + peak
    
    def signature_4(freqs):
        """Multiple small peaks (harmonics)"""
        baseline = -80 * np.ones_like(freqs)
        harmonics = np.zeros_like(freqs)
        for i, f_center in enumerate([1000, 2000, 3000, 4000], 1):
            harmonics += (30 / i) * np.exp(-((freqs - f_center)**2) / (40**2))
        return baseline + harmonics
    
    def signature_5(freqs):
        """Wide-band elevated region"""
        baseline = -80 * np.ones_like(freqs)
        elevated = 20 * (np.tanh((freqs - 2000) / 200) - np.tanh((freqs - 4000) / 200))
        return baseline + elevated
    
    return frequencies, [signature_1, signature_2, signature_3, signature_4, signature_5]


def generate_psd_dataset(n_samples: int, 
                        n_frequencies: int = 14000,
                        noise_level: float = 0.0,
                        signature_distribution: List[float] = None) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Generate a synthetic dataset of PSD measurements.
    
    Args:
        n_samples: Number of PSD samples to generate
        n_frequencies: Number of frequency points per PSD
        noise_level: Standard deviation of Gaussian noise to add (in dB)
        signature_distribution: Probability distribution over signatures (must sum to 1)
        
    Returns:
        psd_data: Array of shape (n_samples, n_frequencies) with PSD values
        frequencies: Array of frequency values
        labels: Ground truth labels for which signature was used
    """
    frequencies, signature_funcs = generate_synthetic_psd_signatures(n_frequencies)
    n_signatures = len(signature_funcs)
    
    if signature_distribution is None:
        signature_distribution = np.ones(n_signatures) / n_signatures
    
    # Generate samples
    psd_data = np.zeros((n_samples, n_frequencies))
    labels = np.random.choice(n_signatures, size=n_samples, p=signature_distribution)
    
    for i in range(n_samples):
        # Generate base signature
        sig_func = signature_funcs[labels[i]]
        psd_data[i] = sig_func(frequencies)
        
        # Add noise if specified
        if noise_level > 0:
            psd_data[i] += np.random.normal(0, noise_level, n_frequencies)
    
    return psd_data, frequencies, labels


def apply_pca(data: np.ndarray, n_components: int = 10, standardize: bool = True) -> Tuple[np.ndarray, PCA, StandardScaler]:
    """
    Apply PCA to reduce dimensionality of PSD data.
    
    Args:
        data: Array of shape (n_samples, n_features)
        n_components: Number of principal components to keep
        standardize: Whether to standardize data before PCA
        
    Returns:
        transformed_data: PCA-transformed data
        pca: Fitted PCA object
        scaler: Fitted StandardScaler object (or None if standardize=False)
    """
    scaler = None
    data_scaled = data
    
    if standardize:
        scaler = StandardScaler()
        data_scaled = scaler.fit_transform(data)
    
    pca = PCA(n_components=n_components)
    transformed_data = pca.fit_transform(data_scaled)
    
    print(f"PCA completed:")
    print(f"  Original dimensions: {data.shape[1]}")
    print(f"  Reduced dimensions: {n_components}")
    print(f"  Explained variance ratio (total): {pca.explained_variance_ratio_.sum():.4f}")
    print(f"  Top 3 components variance: {pca.explained_variance_ratio_[:3]}")
    
    return transformed_data, pca, scaler


def apply_dbscan(data: np.ndarray, eps: float = 0.5, min_samples: int = 5) -> Tuple[np.ndarray, DBSCAN]:
    """
    Apply DBSCAN clustering to PCA-reduced data.
    
    Args:
        data: PCA-transformed data
        eps: Maximum distance between two samples for one to be considered in the neighborhood of the other
        min_samples: Minimum number of samples in a neighborhood for a point to be considered a core point
        
    Returns:
        cluster_labels: Cluster labels (-1 for noise/outliers)
        dbscan: Fitted DBSCAN object
    """
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    cluster_labels = dbscan.fit_predict(data)
    
    n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
    n_noise = list(cluster_labels).count(-1)
    
    print(f"\nDBSCAN clustering completed:")
    print(f"  Number of clusters: {n_clusters}")
    print(f"  Number of noise points: {n_noise}")
    print(f"  Cluster sizes: {np.bincount(cluster_labels[cluster_labels >= 0])}")
    
    return cluster_labels, dbscan


def plot_pca_variance(pca: PCA, n_components_to_show: int = 20):
    """
    Plot explained variance ratio for PCA components.
    """
    n_show = min(n_components_to_show, len(pca.explained_variance_ratio_))
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    
    # Individual variance
    ax1.bar(range(n_show), pca.explained_variance_ratio_[:n_show])
    ax1.set_xlabel('Principal Component')
    ax1.set_ylabel('Explained Variance Ratio')
    ax1.set_title('Variance Explained by Each Component')
    ax1.grid(True, alpha=0.3)
    
    # Cumulative variance
    cumsum_variance = np.cumsum(pca.explained_variance_ratio_[:n_show])
    ax2.plot(range(n_show), cumsum_variance, marker='o')
    ax2.axhline(y=0.95, color='r', linestyle='--', label='95% variance')
    ax2.set_xlabel('Number of Components')
    ax2.set_ylabel('Cumulative Explained Variance Ratio')
    ax2.set_title('Cumulative Variance Explained')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()


def plot_clusters_2d(pca_data: np.ndarray, 
                     cluster_labels: np.ndarray, 
                     true_labels: np.ndarray = None,
                     title: str = "DBSCAN Clusters in PCA Space"):
    """
    Plot clusters in 2D PCA space (first two components).
    """
    fig, axes = plt.subplots(1, 2 if true_labels is not None else 1, figsize=(14 if true_labels is not None else 8, 5))
    if true_labels is None:
        axes = [axes]
    
    # Plot DBSCAN clusters
    unique_clusters = set(cluster_labels)
    colors = plt.cm.rainbow(np.linspace(0, 1, len(unique_clusters)))
    
    for cluster_id, color in zip(sorted(unique_clusters), colors):
        mask = cluster_labels == cluster_id
        label = f'Noise' if cluster_id == -1 else f'Cluster {cluster_id}'
        marker = 'x' if cluster_id == -1 else 'o'
        alpha = 0.3 if cluster_id == -1 else 0.6
        
        axes[0].scatter(pca_data[mask, 0], pca_data[mask, 1], 
                       c=[color], label=label, marker=marker, alpha=alpha, s=50)
    
    axes[0].set_xlabel('PC1')
    axes[0].set_ylabel('PC2')
    axes[0].set_title(title)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Plot true labels if provided
    if true_labels is not None:
        unique_true = set(true_labels)
        colors_true = plt.cm.rainbow(np.linspace(0, 1, len(unique_true)))
        
        for true_label, color in zip(sorted(unique_true), colors_true):
            mask = true_labels == true_label
            axes[1].scatter(pca_data[mask, 0], pca_data[mask, 1],
                          c=[color], label=f'Signature {true_label}', alpha=0.6, s=50)
        
        axes[1].set_xlabel('PC1')
        axes[1].set_ylabel('PC2')
        axes[1].set_title('True Signature Labels')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()


def plot_representative_traces(psd_data: np.ndarray,
                               frequencies: np.ndarray,
                               cluster_labels: np.ndarray,
                               n_examples: int = 3):
    """
    Plot representative PSD traces from each cluster.
    """
    unique_clusters = sorted(set(cluster_labels))
    n_clusters = len(unique_clusters)
    
    fig, axes = plt.subplots(n_clusters, 1, figsize=(14, 4 * n_clusters))
    if n_clusters == 1:
        axes = [axes]
    
    for idx, cluster_id in enumerate(unique_clusters):
        ax = axes[idx]
        mask = cluster_labels == cluster_id
        cluster_data = psd_data[mask]
        
        # Get representative samples (random selection)
        n_to_plot = min(n_examples, len(cluster_data))
        indices = np.random.choice(len(cluster_data), n_to_plot, replace=False)
        
        # Plot individual traces
        for i in indices:
            ax.plot(frequencies, cluster_data[i], alpha=0.4, linewidth=1)
        
        # Plot mean trace
        mean_trace = np.mean(cluster_data, axis=0)
        ax.plot(frequencies, mean_trace, 'k-', linewidth=2, label='Mean')
        
        # Plot std envelope
        std_trace = np.std(cluster_data, axis=0)
        ax.fill_between(frequencies, mean_trace - std_trace, mean_trace + std_trace,
                        alpha=0.2, color='gray', label='±1 std')
        
        label = f'Noise/Outliers (n={len(cluster_data)})' if cluster_id == -1 else f'Cluster {cluster_id} (n={len(cluster_data)})'
        ax.set_title(label)
        ax.set_xlabel('Frequency (MHz)')
        ax.set_ylabel('Power (dBm)')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()


def plot_sample_psds(psd_data: np.ndarray, 
                    frequencies: np.ndarray, 
                    labels: np.ndarray,
                    n_samples_per_sig: int = 2,
                    title: str = "Sample PSD Traces by Signature"):
    """
    Plot sample PSDs for each signature type.
    """
    unique_sigs = sorted(set(labels))
    n_sigs = len(unique_sigs)
    
    fig, axes = plt.subplots(n_sigs, 1, figsize=(14, 3 * n_sigs))
    if n_sigs == 1:
        axes = [axes]
    
    for idx, sig in enumerate(unique_sigs):
        mask = labels == sig
        sig_data = psd_data[mask]
        
        # Plot a few examples
        n_to_plot = min(n_samples_per_sig, len(sig_data))
        sample_indices = np.random.choice(len(sig_data), n_to_plot, replace=False)
        
        for i in sample_indices:
            axes[idx].plot(frequencies, sig_data[i], alpha=0.7, linewidth=1.5)
        
        axes[idx].set_title(f'Signature {sig} (n={len(sig_data)} total)')
        axes[idx].set_xlabel('Frequency (MHz)')
        axes[idx].set_ylabel('Power (dBm)')
        axes[idx].grid(True, alpha=0.3)
    
    fig.suptitle(title, fontsize=14, y=1.00)
    plt.tight_layout()
    plt.show()

---
# Example 1: Clean Synthetic Data

Generate clean synthetic PSD data with 5 distinct signatures and apply PCA + DBSCAN clustering.

## Generate Clean Dataset

In [ ]:
# Parameters for clean data
N_SAMPLES_CLEAN = 1000  # Using 1000 samples for faster computation (scale to 8000 later)
N_FREQUENCIES = 14000
NOISE_LEVEL_CLEAN = 0.0  # No noise for clean example

# Generate dataset
print("Generating clean synthetic PSD dataset...")
psd_data_clean, frequencies, true_labels_clean = generate_psd_dataset(
    n_samples=N_SAMPLES_CLEAN,
    n_frequencies=N_FREQUENCIES,
    noise_level=NOISE_LEVEL_CLEAN
)

print(f"\nDataset shape: {psd_data_clean.shape}")
print(f"Frequency range: {frequencies[0]:.1f} - {frequencies[-1]:.1f} MHz")
print(f"Number of unique signatures: {len(set(true_labels_clean))}")

## Visualize Sample PSDs

In [ ]:
plot_sample_psds(psd_data_clean, frequencies, true_labels_clean, 
                n_samples_per_sig=3, title="Clean Synthetic PSD Signatures")

## Apply PCA

In [ ]:
# Apply PCA with 10 components
N_COMPONENTS = 10

pca_data_clean, pca_clean, scaler_clean = apply_pca(
    psd_data_clean, 
    n_components=N_COMPONENTS,
    standardize=True
)

In [ ]:
# Visualize PCA variance
plot_pca_variance(pca_clean, n_components_to_show=10)

## Apply DBSCAN Clustering

In [ ]:
# Apply DBSCAN
# Note: These parameters may need tuning based on the data
EPS_CLEAN = 3.0
MIN_SAMPLES_CLEAN = 10

cluster_labels_clean, dbscan_clean = apply_dbscan(
    pca_data_clean,
    eps=EPS_CLEAN,
    min_samples=MIN_SAMPLES_CLEAN
)

## Visualize Clusters

In [ ]:
# Plot clusters in PCA space
plot_clusters_2d(pca_data_clean, cluster_labels_clean, true_labels_clean,
                title="DBSCAN Clusters (Clean Data)")

## Plot Representative Traces from Each Cluster

In [ ]:
# Plot representative PSD traces for each cluster
plot_representative_traces(psd_data_clean, frequencies, cluster_labels_clean, n_examples=5)

## Cluster Quality Analysis

In [ ]:
# Compare cluster assignments to true labels
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

# Filter out noise points for fair comparison
non_noise_mask = cluster_labels_clean >= 0

if non_noise_mask.sum() > 0:
    ari = adjusted_rand_score(true_labels_clean[non_noise_mask], cluster_labels_clean[non_noise_mask])
    nmi = normalized_mutual_info_score(true_labels_clean[non_noise_mask], cluster_labels_clean[non_noise_mask])
    
    print("Clustering Quality Metrics (excluding noise points):")
    print(f"  Adjusted Rand Index: {ari:.4f} (1.0 = perfect match)")
    print(f"  Normalized Mutual Information: {nmi:.4f} (1.0 = perfect match)")
else:
    print("All points classified as noise - try adjusting DBSCAN parameters")

---
# Example 2: Noisy Synthetic Data

Generate synthetic PSD data with added Gaussian noise to test robustness of the approach.

## Generate Noisy Dataset

In [ ]:
# Parameters for noisy data
N_SAMPLES_NOISY = 1000
NOISE_LEVEL_NOISY = 2.0  # 2 dB standard deviation of Gaussian noise

# Generate dataset
print("Generating noisy synthetic PSD dataset...")
psd_data_noisy, _, true_labels_noisy = generate_psd_dataset(
    n_samples=N_SAMPLES_NOISY,
    n_frequencies=N_FREQUENCIES,
    noise_level=NOISE_LEVEL_NOISY
)

print(f"\nDataset shape: {psd_data_noisy.shape}")
print(f"Noise level: {NOISE_LEVEL_NOISY} dB")
print(f"Number of unique signatures: {len(set(true_labels_noisy))}")

## Visualize Sample Noisy PSDs

In [ ]:
plot_sample_psds(psd_data_noisy, frequencies, true_labels_noisy, 
                n_samples_per_sig=3, title="Noisy Synthetic PSD Signatures")

## Apply PCA

In [ ]:
# Apply PCA with same number of components
pca_data_noisy, pca_noisy, scaler_noisy = apply_pca(
    psd_data_noisy, 
    n_components=N_COMPONENTS,
    standardize=True
)

In [ ]:
# Visualize PCA variance
plot_pca_variance(pca_noisy, n_components_to_show=10)

## Apply DBSCAN Clustering

In [ ]:
# Apply DBSCAN - may need different parameters due to noise
EPS_NOISY = 4.0  # Larger eps to account for noise-induced spread
MIN_SAMPLES_NOISY = 10

cluster_labels_noisy, dbscan_noisy = apply_dbscan(
    pca_data_noisy,
    eps=EPS_NOISY,
    min_samples=MIN_SAMPLES_NOISY
)

## Visualize Clusters

In [ ]:
# Plot clusters in PCA space
plot_clusters_2d(pca_data_noisy, cluster_labels_noisy, true_labels_noisy,
                title="DBSCAN Clusters (Noisy Data)")

## Plot Representative Traces from Each Cluster

In [ ]:
# Plot representative PSD traces for each cluster
plot_representative_traces(psd_data_noisy, frequencies, cluster_labels_noisy, n_examples=5)

## Cluster Quality Analysis

In [ ]:
# Compare cluster assignments to true labels
non_noise_mask_noisy = cluster_labels_noisy >= 0

if non_noise_mask_noisy.sum() > 0:
    ari_noisy = adjusted_rand_score(true_labels_noisy[non_noise_mask_noisy], 
                                    cluster_labels_noisy[non_noise_mask_noisy])
    nmi_noisy = normalized_mutual_info_score(true_labels_noisy[non_noise_mask_noisy], 
                                             cluster_labels_noisy[non_noise_mask_noisy])
    
    print("Clustering Quality Metrics (excluding noise points):")
    print(f"  Adjusted Rand Index: {ari_noisy:.4f} (1.0 = perfect match)")
    print(f"  Normalized Mutual Information: {nmi_noisy:.4f} (1.0 = perfect match)")
    
    print("\nComparison to Clean Data:")
    if 'ari' in locals():
        print(f"  ARI degradation: {ari - ari_noisy:.4f}")
        print(f"  NMI degradation: {nmi - nmi_noisy:.4f}")
else:
    print("All points classified as noise - try adjusting DBSCAN parameters")

---
# Summary and Next Steps

## Key Observations

1. **PCA Effectiveness**: Check how much variance is captured by the first few components
2. **Cluster Quality**: Compare DBSCAN clusters to true signatures using ARI and NMI
3. **Noise Robustness**: Evaluate how noise affects clustering performance

## Potential Improvements

1. **Hyperparameter Tuning**: 
   - Optimize number of PCA components
   - Grid search for optimal DBSCAN eps and min_samples

2. **Alternative Approaches**:
   - Try other clustering algorithms (K-means, GMM, HDBSCAN)
   - Explore other dimensionality reduction techniques (t-SNE, UMAP)
   - Consider autoencoders for non-linear dimensionality reduction

3. **Anomaly Detection**:
   - Use isolation forest or one-class SVM for outlier detection
   - Implement reconstruction error from PCA for anomaly scoring

4. **Scale to Real Data**:
   - Test with 8000 samples
   - Apply to real RF measurements
   - Add temporal analysis (how signatures change over time)